# MinerU PDF Ingestion — 6.0001 Introduction to Python
Two-stage pipeline: PDF pages → MinerU markdown → structured JSON problems.

In [9]:
!uv pip install --upgrade mlx-vlm pymupdf pillow

Resolved 67 packages in 136ms                                        
Audited 67 packages in 0.96ms


In [10]:
import os, re, zipfile, urllib.request
from pathlib import Path

COURSE_SLUG = "6-0001-introduction-to-computer-science-and-programming-in-python-fall-2016"
BASE_URL = f"https://ocw.mit.edu/courses/{COURSE_SLUG}/"
WORK_DIR = Path("/tmp/ocw_ingestion")
EXTRACT_DIR = WORK_DIR / COURSE_SLUG

WORK_DIR.mkdir(parents=True, exist_ok=True)

if EXTRACT_DIR.exists():
    print(f"Already extracted: {EXTRACT_DIR}")
else:
    download_page_url = BASE_URL + "download"
    print(f"Fetching {download_page_url}")
    with urllib.request.urlopen(download_page_url) as resp:
        html = resp.read().decode("utf-8")

    match = re.search(r'href="([^"]+\.zip)"', html)
    if not match:
        raise RuntimeError("No .zip link found on download page")
    zip_href = match.group(1)
    if not zip_href.startswith("http"):
        zip_href = "https://ocw.mit.edu" + zip_href
    print(f"Zip URL: {zip_href}")

    zip_path = WORK_DIR / f"{COURSE_SLUG}.zip"
    print(f"Downloading to {zip_path} ...")
    urllib.request.urlretrieve(zip_href, zip_path)
    print(f"Downloaded ({zip_path.stat().st_size / 1e6:.1f} MB)")

    print("Extracting ...")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}")

Already extracted: /tmp/ocw_ingestion/6-0001-introduction-to-computer-science-and-programming-in-python-fall-2016


In [11]:
import re
from pathlib import Path

EXTRACT_DIR = Path("/tmp/ocw_ingestion") / "6-0001-introduction-to-computer-science-and-programming-in-python-fall-2016"

static_resources = EXTRACT_DIR / "static_resources"
all_pdfs = sorted(static_resources.glob("*.pdf"))

SOL_PATTERN = re.compile(r"sol|solution|ans", re.IGNORECASE)

question_pdfs = [p for p in all_pdfs if not SOL_PATTERN.search(p.name)]
solution_pdfs = [p for p in all_pdfs if SOL_PATTERN.search(p.name)]

print(f"Total PDFs: {len(all_pdfs)}")
print(f"\nQuestion PDFs ({len(question_pdfs)}):")
for p in question_pdfs:
    print(f"  {p.name}")
print(f"\nSolution PDFs ({len(solution_pdfs)}):")
for p in solution_pdfs:
    print(f"  {p.name}")

TARGET_PDF = question_pdfs[0]
print(f"\nTarget PDF: {TARGET_PDF}")

Total PDFs: 81

Question PDFs (80):
  03c0992b7c0cf9d74884b45e19a5b5f7_vqn_yk5aFcI.pdf
  04a7dce7569d2cb05144aab86c1165e7_4WtaFLayz_w.pdf
  066eba6ea6d56a88e56ae325940d4c4c_MIT6_0001F16_Lec10.pdf
  13a7a20e33d6cf30bedf5622c894fd17_qq7I2MQNrtU.pdf
  13ce627607efa9982ae632cd68faed56_w4uxYDPsjbw.pdf
  13f7b896c7548549ae4d5e8f2c5c20b6_P-0w8xWcnDQ.pdf
  1776670e271578eeb99fc25975f20586_MIT6_0001F16_Lec5.pdf
  1a4842e8b4a1b4ab1c2676ce20e6767b_P-0w8xWcnDQ.pdf
  1bbe737a47db4c783c9029578fd82778_mrvBnZIEsZY.pdf
  1c813f689d7cf8b52eb6eae66aab6e13_ax4eNMI9Dw.pdf
  1d65ba61fcc2bd84c648e07109f620c1_6LOwPhPDwVc.pdf
  272cefe6cf36a6b7ab4960209e41b62d_o9nW0uBqvEo.pdf
  29c347446971ec2035a3f0caeb85aa20_zYVWQpCitKQ.pdf
  2cd211c4c7f47153bbc413099ae2d2e1_MjbuarJ7SE0.pdf
  2dd6c75e7b4bd6bd135078e6f3701201_MIT6_0001F16_Lec9.pdf
  2f4aca3461c3939be1a8adb62c0a9a0e_-wz4iU2V-Yo.pdf
  385141ae56a3bd0bbd5cdc1d3437069f_5McjE8e5gIg.pdf
  3b3e814ffe2ef349fac15598b99ee8a1_F-_PKUUM-qY.pdf
  3db3c479431dc5fd4fc841e2e7

In [ ]:
from mlx_vlm import load, generate
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.utils import load_config

MODEL_ID = "mlx-community/MinerU2.5-2509-1.2B-bf16"
print(f"Loading {MODEL_ID} (downloads on first run, cached by HuggingFace)...")
model, processor = load(MODEL_ID)
config = load_config(MODEL_ID)
print("Model loaded.")

In [ ]:
import fitz  # pymupdf
from PIL import Image
import io

def pdf_to_images(pdf_path: str, dpi: int = 150) -> list:
    """Render each PDF page to a PIL Image at the given DPI."""
    doc = fitz.open(pdf_path)
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        images.append(img)
    doc.close()
    return images

pages = pdf_to_images(str(TARGET_PDF))
print(f"Rendered {len(pages)} page(s) from {TARGET_PDF.name}")
print(f"First page size: {pages[0].size}")

In [ ]:
PAGE_TO_MARKDOWN_PROMPT = (
    "Convert this document page to markdown. "
    "Preserve all math using LaTeX: $...$ inline, $$...$$ display. "
    "Reproduce all text exactly."
)

page_markdowns = []

for i, page_img in enumerate(pages):
    formatted_prompt = apply_chat_template(
        processor, config, PAGE_TO_MARKDOWN_PROMPT, num_images=1
    )
    page_md = generate(model, processor, formatted_prompt, [page_img], max_tokens=2048)
    page_markdowns.append(page_md)
    print(f"--- Page {i + 1} ---")
    print(page_md[:300])
    print()

combined_markdown = "\n\n---\n\n".join(page_markdowns)
print("=== Combined Markdown ===")
print(combined_markdown)

In [ ]:
import json, re

page_count = len(pages)
EXTRACTION_PROMPT = f"""Parse the attached QUESTION pages into structured, interactive problems.

The input contains {page_count} page image(s).

Return ONLY a JSON object with this exact structure:
{{
  "content_title": "string|null",
  "problems": [
    {{
      "problem_label": "string",
      "question_text": "string",
      "ordering": 0
    }}
  ]
}}

Rules:
- Group subparts under the parent problem. Example: 1(a), 1(b), 1(c) must be one problem object with one problem_label (e.g. "1").
- Return one array item per top-level problem; the problems array can be any non-negative length.
- Convert math expressions to LaTeX. Use $...$ for inline math and $$...$$ for display math.
- Do not use \\(...\\) or \\[...\\] delimiters.
- Because this is JSON, every LaTeX backslash must be escaped as \\\\ (example: \\\\frac{{a}}{{b}}).
- Exclude cover/instruction pages that are not actual problems.
- ordering must start at 0 and increase in document order.
- If there are no problems, return {{"content_title":null,"problems":[]}}.
- No markdown fences. No commentary."""

formatted_prompt = apply_chat_template(
    processor, config, EXTRACTION_PROMPT + "\n\n" + combined_markdown, num_images=0
)
raw_output = generate(model, processor, formatted_prompt, [], max_tokens=4096)

clean_output = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_output.strip())

try:
    extracted = json.loads(clean_output)
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print("Raw output:")
    print(raw_output)
    extracted = {"content_title": None, "problems": []}

print(f"Title: {extracted.get('content_title')}")
print(f"Problems found: {len(extracted.get('problems', []))}")
print()
for p in extracted.get("problems", []):
    preview = p["question_text"][:120].replace("\n", " ")
    print(f"  [{p['ordering']}] {p['problem_label']}: {preview}...")

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path("/tmp/ocw_ingestion")
pdf_stem = TARGET_PDF.stem
output_path = OUTPUT_DIR / f"problems_{pdf_stem}.json"

with open(output_path, "w") as f:
    json.dump(extracted, f, indent=2, ensure_ascii=False)

print(f"Saved {len(extracted.get('problems', []))} problems to {output_path}")
print()
if extracted.get("problems"):
    first = extracted["problems"][0]
    print("First problem preview:")
    print(f"  label:    {first['problem_label']}")
    print(f"  ordering: {first['ordering']}")
    print(f"  text:     {first['question_text'][:300]}")